In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import joblib
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

df = pd.read_csv('heart.csv')

print(df.head())

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'L1': LogisticRegression(penalty='l1', solver='saga', max_iter=5000, random_state=SEED),
    'L2': LogisticRegression(penalty='l2', solver='lbfgs', max_iter=5000, random_state=SEED),
    'ElasticNet': LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, l1_ratio=0.5, random_state=SEED),
}

for name, model in models.items():
    print(f"\nTraining {name} Logistic Regression...")
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    print(f"--- {name} Regularization ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")
    print(classification_report(y_test, y_pred))


   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  

Training L1 Logistic Regression...
--- L1 Regularization ---
Accuracy: 0.8033
ROC AUC: 0.8701
              precision    recall  f1-score   support

           0       0.86      0.68      0.76        28
           1       0.77      0.91      0.83        33

    accuracy                           0.80        61
   macro avg       0.82      0.79      0.80        61
weighted avg  

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import random
import json
import os

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

df = pd.read_csv('student_scores_v1.csv')
print("Dataset preview:")
print(df.head())

X = df[['Hours_Studied']]
y = df['Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nModel Evaluation:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/poly_reg_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')
joblib.dump(poly, 'models/poly_features.joblib')

metadata = {
    'seed': SEED,
    'polynomial_degree': 2,
    'test_size': 0.2,
    'mse': mse,
    'r2': r2,
}

with open('models/experiment_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("\nModel, scaler, polynomial features, and metadata saved in 'models/' folder.")


Dataset preview:
   Hours_Studied  Score
0              1     10
1              2     25
2              3     35
3              4     50
4              5     65

Model Evaluation:
Mean Squared Error: 0.6272
R^2 Score: 0.9995

Model, scaler, polynomial features, and metadata saved in 'models/' folder.


In [1]:
import numpy as np
import pandas as pd
import random
import os
import json
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Features: {feature_names}")
print(f"Target classes: {target_names}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

os.makedirs('knn_models', exist_ok=True)

results = []

for k in [1, 3, 5, 7, 9]:
    print(f"\nTraining KNN with K={k}")
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy for K={k}: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=target_names))

    model_path = f'knn_models/knn_k{k}.joblib'
    joblib.dump(knn, model_path)

    results.append({
        'k': k,
        'accuracy': acc,
        'model_path': model_path,
    })

with open('knn_models/experiment_summary.json', 'w') as f:
    json.dump(results, f, indent=4)

print("\nModels and experiment summary saved in 'knn_models/' directory.")


Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target classes: ['setosa' 'versicolor' 'virginica']

Training KNN with K=1
Accuracy for K=1: 0.9667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.91      1.00      0.95        10
   virginica       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


Training KNN with K=3
Accuracy for K=3: 1.0000
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00        10
   virginica       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        3